## Data Loader and example usage of learnt models

In [21]:
import os
import scipy.io
from PIL import Image

import torch
import torch.nn as nn

from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from torchvision import transforms
from tqdm import tqdm

In [22]:
## Device
device=torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)
print(device)

cpu


In [23]:
### Dataset Class
class FlowersDataset(Dataset):
    def __init__(self, image_dir, labels_path, setid_path, split="train", transform=None):
        self.image_dir=image_dir
        self.transform=transform
        
        labels_data=scipy.io.loadmat(labels_path)
        setid_data=scipy.io.loadmat(setid_path)
        
        self.labels=labels_data["labels"][0]
        
        if split=="train":
            self.indices=setid_data["trnid"][0]
        elif split=="valid":
            self.indices=setid_data["valid"][0]
        elif split=="test":
            self.indices=setid_data["tstid"][0]
        else:
            raise ValueError(
                "Split must be train, valid or test"
            )
            
    def __len__(self):
        return len(self.indices)
    
    def __getitem__(self, idx):
        img_id=self.indices[idx]
        img_name=f"image_{img_id:05d}.jpg"
        
        img_path=os.path.join(
            self.image_dir, img_name
        )
        
        image=Image.open(img_path).convert("RGB")
        
        label=self.labels[img_id-1]-1
        
        if self.transform:
            image=self.transform(image)
        
        return image, label
    

In [24]:
## Image Transformations

transform=transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [25]:
ROOT = "../DB/102flowers"

train_dataset = FlowersDataset(
    image_dir=f"{ROOT}/jpg",
    labels_path=f"{ROOT}/imagelabels.mat",
    setid_path=f"{ROOT}/setid.mat",
    split="train",
    transform=transform
)

valid_dataset = FlowersDataset(
    image_dir=f"{ROOT}/jpg",
    labels_path=f"{ROOT}/imagelabels.mat",
    setid_path=f"{ROOT}/setid.mat",
    split="valid",
    transform=transform
)

test_dataset = FlowersDataset(
    image_dir=f"{ROOT}/jpg",
    labels_path=f"{ROOT}/imagelabels.mat",
    setid_path=f"{ROOT}/setid.mat",
    split="test",
    transform=transform
)

In [26]:
print("Train:", len(train_dataset))
print("Valid:", len(valid_dataset))
print("Test :", len(test_dataset))

Train: 1020
Valid: 1020
Test : 6149


In [27]:
## Create DataLoaders
train_loader=DataLoader(
    train_dataset, batch_size=32, shuffle=True
)

valid_loader=DataLoader(
    valid_dataset, batch_size=32,
    shuffle=False
)

test_loader=DataLoader(
    test_dataset, batch_size=32,
    shuffle=False
)

In [28]:
## Check Data
images, labels=next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([32, 3, 224, 224])
torch.Size([32])


In [29]:
# Convert DataLoader batches to TensorFlow format

import numpy as np

def dataloader_to_numpy(loader):
    images_list=[]
    labels_list=[]
    
    for images, labels in loader:
        #NCHW->NHWC
        images=images.permute(0, 2, 3, 1)
        
        images_list.append(images.numpy())
        labels_list.append(labels.numpy())
        
    X=np.concatenate(images_list, axis=0)
    y=np.concatenate(labels_list, axis=0)
    
    return X, y

In [30]:
## Create Tensorflow Datasets
X_train, y_train = dataloader_to_numpy(train_loader)

X_valid, y_valid = dataloader_to_numpy(valid_loader)

X_test, y_test = dataloader_to_numpy(test_loader)

print(X_train.shape)
print(y_train.shape)

(1020, 224, 224, 3)
(1020,)


In [31]:
## Import TensorFlow
import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.models import Model

from tensorflow.keras.layers import (
    Input,
    Conv2D,
    MaxPooling2D,
    Flatten,
    Dense,
    Dropout
)

In [32]:
## Model 1 ---- Sequential API

seq_model=Sequential([
    Conv2D(32, (3,3), activation="relu", input_shape=(224, 224, 3)),
    MaxPooling2D(),
    
    Conv2D(64, (3, 3), activation="relu"),
    MaxPooling2D(),
    
    Conv2D(128, (3, 3), activation="relu"),
    MaxPooling2D(),
    
    Flatten(),
    Dense(512, activation="relu"),
    
    Dropout(0.5),
    
    Dense(102, activation="softmax")
    
])

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [33]:
## Compile 
seq_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

### Train 
history_seq=seq_model.fit(
    X_train, y_train, validation_data=(X_valid, y_valid),
    epochs=10, batch_size=32
)

Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 43s 1s/step - accuracy: 0.0078 - loss: 4.8022 - val_accuracy: 0.0284 - val_loss: 4.4349
Epoch 2/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 35s 1s/step - accuracy: 0.0324 - loss: 4.3466 - val_accuracy: 0.0529 - val_loss: 4.1363
Epoch 3/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 30s 943ms/step - accuracy: 0.0784 - loss: 3.9845 - val_accuracy: 0.1059 - val_loss: 3.8692
Epoch 4/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 31s 953ms/step - accuracy: 0.1892 - loss: 3.4374 - val_accuracy: 0.1284 - val_loss: 3.7297
Epoch 5/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 29s 906ms/step - accuracy: 0.4098 - loss: 2.3403 - val_accuracy: 0.1343 - val_loss: 3.7882
Epoch 6/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 0.6490 - loss: 1.3367 - val_accuracy: 0.1216 - val_loss: 4.3865
Epoch 7/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 30s 943ms/step - accuracy: 0.8186 - loss: 0.6391 - val_accuracy: 0.1431 - val_loss: 4.7444
Epoch 8/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 28s 859ms/step - accuracy: 0.8863 - loss: 0.4261 - val_accuracy: 0.1

In [37]:
# Evaluate
seq_model.evaluate(
    X_test, y_test
)

193/193 ━━━━━━━━━━━━━━━━━━━━ 30s 144ms/step - accuracy: 0.1218 - loss: 5.8160


[5.815957069396973, 0.12180842459201813]

## Functional API

In [35]:
## Functional API
inputs=Input(shape=(224, 224, 3))

x=Conv2D(
    32, (3, 3), activation="relu"
)(inputs)

x=MaxPooling2D()(x)

x=Conv2D(
    64, (3, 3), activation="relu"
)(x)

x=MaxPooling2D()(x)

x=Conv2D(128, (3, 3), activation="relu")(x)

x=MaxPooling2D()(x)
x=Flatten()(x)

x=Dense(512, activation='relu')(x)

x=Dropout(0.5)(x)
outputs=Dense(
    102, activation="softmax"
)(x)

func_model=Model(inputs, outputs)

In [36]:
# Compile
func_model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_func=func_model.fit(
    X_train, y_train, validation_data=(X_valid, y_valid),
    epochs=10, batch_size=32
)

Epoch 1/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.0098 - loss: 4.7393 - val_accuracy: 0.0314 - val_loss: 4.5667
Epoch 2/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 0.0275 - loss: 4.4210 - val_accuracy: 0.0480 - val_loss: 4.1880
Epoch 3/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 30s 918ms/step - accuracy: 0.0686 - loss: 4.0203 - val_accuracy: 0.0882 - val_loss: 3.9366
Epoch 4/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 38s 1s/step - accuracy: 0.1176 - loss: 3.7045 - val_accuracy: 0.1353 - val_loss: 3.7566
Epoch 5/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 44s 1s/step - accuracy: 0.2873 - loss: 2.9513 - val_accuracy: 0.1549 - val_loss: 3.5604
Epoch 6/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 44s 1s/step - accuracy: 0.5235 - loss: 1.8228 - val_accuracy: 0.1657 - val_loss: 3.8647
Epoch 7/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 36s 1s/step - accuracy: 0.7235 - loss: 1.0448 - val_accuracy: 0.1735 - val_loss: 4.0336
Epoch 8/10
32/32 ━━━━━━━━━━━━━━━━━━━━ 34s 1s/step - accuracy: 0.8451 - loss: 0.5909 - val_accuracy: 0.1725 - val_lo